# Tien xu ly CQT Cache FP16 (Google Colab GPU T4) cho QBH V6.1

Notebook nay chay tren **Google Colab (GPU T4)** de trich xuat truoc toan bo dac trung CQT (FP16) cho tap huan luyen & validation cua mo hinh **Conformer-ByteHum V6.1**.

### Muc tieu va loi ich:
- **Tiet kiem thoi gian tren Kaggle:** Giam 10 - 15 phut giai ma MP3/WAV va tinh CQT o moi phien chay train.
- **Tiet kiem RAM CPU:** Toan bo tensor CQT duoc luu o dinh dang nua chinh xac `torch.float16`.
- **Dong bo 100% ma bam CQT Backend:** Khop chinh xac `CQT_BACKEND_SHA256 = '01fda8163329110fca01d0529fcb13324b7156e616b71533bb29e720ace2ec05'` ma notebook train V6.1 yeu cau.
- **Xuat ra thu muc xuat ban:** Sau khi chay xong, toan bo du lieu cache duoc xuat ra folder rieng de de dang tai ve may hoac sao luu vao Google Drive.

### Quy trinh thuc hien:
1. Vao **Runtime** -> **Change runtime type** -> Chon **T4 GPU** -> **Save**.
2. Chay tuan tu cac o ben duoi:
   - Ket noi Google Drive (hoac upload truc tiep file `datasets_qbh_clean_mirqbsh_70_15_15_v6_1.zip`).
   - Giai nen dataset.
   - Chay trich xuat CQT bang CUDA GPU (khoang 5 - 8 phut).
   - Dong goi ket qua vao folder xuat ban `export_qbh_cqt_cache_v6_1` va file zip de tai ve may.
3. Upload file ZIP cache len Kaggle Dataset de lam Input cho notebook huan luyen.

In [ ]:
# ==========================================================
# 1. KIEM TRA MOI TRUONG GPU VA CAI DAT THU VIEN
# ==========================================================
import os, sys, subprocess, importlib.util
import torch

print('[INFO] Python Version :', sys.version.split()[0])
print('[INFO] PyTorch Ver    :', torch.__version__)

if not torch.cuda.is_available():
    raise RuntimeError('[ERROR] CHUA BAT GPU! Vao Runtime -> Change runtime type -> Chon T4 GPU -> Nhan Save.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f'[INFO] GPU Hardware   : {gpu_name} ({vram_gb} GB VRAM)')

# Cai dat thu vien am thanh neu chua co
deps = ['librosa', 'soundfile', 'scipy', 'tqdm']
for dep in deps:
    if importlib.util.find_spec(dep) is None:
        print(f'[INSTALL] Cai dat goi: {dep}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', dep])
print('[OK] Tat ca thu vien phu thuoc da san sang!')

In [ ]:
# ==========================================================
# 2. KET NOI GOOGLE DRIVE (TUY CHON)
# ==========================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print('[INFO] Google Drive da duoc gan thanh cong tai /content/drive')
except Exception as e:
    DRIVE_MOUNTED = False
    print('[INFO] Khong dung Google Drive hoac chay ngoai Colab. Tiep tuc voi bo nho cuc bo /content.')

In [ ]:
# ==========================================================
# 3. DINH VI VA GIAI NEN DATASET V6.1
# ==========================================================
import shutil, zipfile
from pathlib import Path

DATASET_ZIP_NAME = 'datasets_qbh_clean_mirqbsh_70_15_15_v6_1.zip'
WORKSPACE_DIR = Path('/content')
EXTRACTED_DIR = WORKSPACE_DIR / 'datasets_extracted'

# Tu dong tim kiem file ZIP trong Google Drive hoac /content
search_dirs = [WORKSPACE_DIR]
if DRIVE_MOUNTED:
    search_dirs.extend([
        Path('/content/drive/MyDrive'),
        Path('/content/drive/MyDrive/QBH'),
        Path('/content/drive/MyDrive/PBL6')
    ])

zip_path = None
for s_dir in search_dirs:
    cand = s_dir / DATASET_ZIP_NAME
    if cand.is_file():
        zip_path = cand
        break

if zip_path is None:
    zip_path = WORKSPACE_DIR / DATASET_ZIP_NAME
    if not zip_path.is_file():
        print(f'[WARNING] Chua tim thay {DATASET_ZIP_NAME} trong Google Drive hay /content.')
        print(f'[HUONG DAN] Keo tha file {DATASET_ZIP_NAME} vao tab Files ben trai Colab (tai /content).')
        raise FileNotFoundError(f'Vui long tai file {DATASET_ZIP_NAME} vao {WORKSPACE_DIR} hoac Google Drive.')

print(f'[INFO] Tim thay dataset ZIP tai: {zip_path}')
print(f'[INFO] Dung luong file: {round(zip_path.stat().st_size / (1024**3), 2)} GB')

# Giai nen dataset
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
print('[INFO] Dang giai nen dataset... (khoang 30 - 60 giay)')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACTED_DIR)

if (EXTRACTED_DIR / 'datasets').is_dir():
    DATASET_ROOT = EXTRACTED_DIR / 'datasets'
else:
    DATASET_ROOT = EXTRACTED_DIR

print(f'[OK] Giai nen hoan tat! DATASET_ROOT = {DATASET_ROOT}')
assert (DATASET_ROOT / 'dataset_split').is_dir(), 'Thieu thu muc dataset_split'
assert (DATASET_ROOT / '2_music_database').is_dir(), 'Thieu thu muc 2_music_database'
assert (DATASET_ROOT / 'qbh_dataset_contract.json').is_file(), 'Thieu qbh_dataset_contract.json'

In [ ]:
# ==========================================================
# 4. DINH NGHIA BO TRICH XUAT CQT CHUAN V6.1 TREN CUDA
# ==========================================================
import math, hashlib, scipy.sparse
import numpy as np
import torch.nn.functional as F
import librosa, soundfile as sf

SAMPLE_RATE = 16000
CHUNK_SAMPLES = 10 * SAMPLE_RATE
HOP_SAMPLES = 3 * SAMPLE_RATE

PREPROCESS_CONFIG = dict(sr=16000, chunk_seconds=10, hop_seconds=3,
                         final_anchored_window=True, bins=96, frames=83,
                         cqt_hop=0.04, cqt_mean=3, floor_db=-80.0)

class AudioPreprocessor:
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        try:
            y, _ = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        except Exception:
            data, sr_in = sf.read(audio_path, dtype='float32')
            if data.ndim > 1:
                data = data.mean(axis=1)
            if sr_in != self.sample_rate:
                data = librosa.resample(data, orig_sr=sr_in, target_sr=self.sample_rate)
            y = data
        if np.max(np.abs(y)) > 0:
            y = y / (np.max(np.abs(y)) + 1e-8)
        return y

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA is required for GPU CQT.')
        self._hop_size = hop_size
        self._sample_rate = sample_rate
        self._device = torch.device('cuda', torch.cuda.current_device())
        if not max_freq:
            max_freq = sample_rate // 2
        sparse_kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)
        dense_kernel = sparse_kernel.toarray() if scipy.sparse.issparse(sparse_kernel) else np.asarray(sparse_kernel)
        self._kernel = torch.as_tensor(dense_kernel.astype(np.complex64, copy=False), device=self._device)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)
        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel
        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        return np.conjugate(cqt_kernel) / fft_length

    @staticmethod
    @torch.inference_mode()
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel, device):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = cqt_kernel.shape
        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        if number_times <= 0:
            return torch.empty((number_frequencies, 0), dtype=torch.float32, device=device)

        audio_gpu = torch.as_tensor(np.asarray(audio_signal, dtype=np.float32), device=device)
        audio_gpu = F.pad(audio_gpu, (pad_left, pad_right))
        frames = audio_gpu.unfold(0, fft_length, step_length)
        if frames.shape[0] != number_times:
            raise RuntimeError(f'CUDA CQT frame count mismatch: expected {number_times}, got {frames.shape[0]}')

        cqt_gpu = torch.empty((number_frequencies, number_times), dtype=torch.float32, device=device)
        frame_batch = 128
        for start in range(0, number_times, frame_batch):
            end = min(start + frame_batch, number_times)
            spectra = torch.fft.fft(frames[start:end], dim=-1)
            response = torch.matmul(cqt_kernel, spectra.transpose(0, 1))
            cqt_gpu[:, start:end] = response.abs()
        return cqt_gpu

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_gpu = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel, self._device)
        magnitude = cqt_gpu.clamp_min(1e-9)
        ref_value = magnitude.max().clamp_min(1e-4)
        cqt_gpu = (20.0 * torch.log10(magnitude) - 20.0 * torch.log10(ref_value)).clamp(-80.0, 0.0)
        if not feat_dim_first:
            cqt_gpu = cqt_gpu.transpose(0, 1)
        return cqt_gpu.contiguous().cpu().numpy()

class CoverHunterCQT:
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(sample_rate=sample_rate, hop_size=hop_size, octave_resolution=octave_resolution,
                            min_freq=min_freq, max_freq=max_freq)

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()

class CQTExtractor:
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(sample_rate=sample_rate, hop_size=hop_size,
                                      octave_resolution=bins_per_octave, min_freq=32.0, max_freq=8000.0,
                                      mean_size=mean_size)

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        feat = self.cqt_impl.extract(y)
        tensor = feat.T.unsqueeze(0)
        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor

device = torch.device('cuda')
prep = AudioPreprocessor()
extractor = CQTExtractor()
EXPECTED_BACKEND_SHA = '01fda8163329110fca01d0529fcb13324b7156e616b71533bb29e720ace2ec05'
print('[OK] CQT Extractor tren GPU T4 da khoi tao thanh cong!')

In [ ]:
# ==========================================================
# 5. CAC HAM TIEN ICH QUET DATASET & HA TAN SO 8 KHZ
# ==========================================================
import csv, json, time
from tqdm.auto import tqdm
import scipy.signal

def digest_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def chunk_starts(length: int):
    if length <= CHUNK_SAMPLES:
        return [0]
    starts = list(range(0, length - CHUNK_SAMPLES + 1, HOP_SAMPLES))
    if starts[-1] + CHUNK_SAMPLES < length:
        starts.append(length - CHUNK_SAMPLES)
    return starts

def low_band_8bit(audio: np.ndarray, sr=16000, seed=0):
    down = scipy.signal.resample_poly(audio, 1, 2).astype(np.float32)
    restored = scipy.signal.resample_poly(down, 2, 1).astype(np.float32)
    rng = np.random.default_rng(seed)
    step = 2.0 / 255.0
    dither = (rng.random(restored.shape) - rng.random(restored.shape)) * step
    return (np.round((np.clip(restored, -1.0, 1.0) + dither) / step) * step).clip(-1.0, 1.0).astype(np.float32)

def get_audio_inventory(dataset_root: Path):
    paths = []
    for rel_root in ('dataset_split/train', 'dataset_split/val',
                     'mir_qbsh_extension/train', 'mir_qbsh_extension/val'):
        folder = dataset_root / rel_root
        if folder.is_dir():
            paths.extend(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.wav', '.flac', '.mp3'})

    mapping_path = dataset_root / 'song_name_mapping.json'
    gallery_dir = dataset_root / '2_music_database'
    id_to_title = json.loads(mapping_path.read_text(encoding='utf-8')) if mapping_path.exists() else {}
    trainval_titles = set()
    for split in ('train', 'val'):
        folder = dataset_root / 'dataset_split' / split
        if folder.exists():
            trainval_titles.update(p.name for p in folder.iterdir() if p.is_dir())
    for path in gallery_dir.glob('*.mp3') if gallery_dir.exists() else []:
        if id_to_title.get(path.stem) in trainval_titles:
            paths.append(path)
    return sorted(set(paths))

def query_source_map(dataset_root: Path):
    path = dataset_root / 'qbh_query_manifest_v6_1.csv'
    if not path.is_file():
        return {}
    with path.open(encoding='utf-8', newline='') as f:
        result = {}
        for row in csv.DictReader(f):
            rel = row['path'].replace('\\', '/')
            if rel.startswith('datasets/'):
                rel = rel[len('datasets/'):]
            result[rel] = row['source']
        return result

inventory = get_audio_inventory(DATASET_ROOT)
source_by_path = query_source_map(DATASET_ROOT)
print(f'[INFO] Tim thay tong cong {len(inventory)} file am thanh (train, val va gallery tuong ung).')
print('[INFO] Mau test va locked-holdout hoan toan duoc bo qua de giu sach tuyet doi.')

In [ ]:
# ==========================================================
# 6. TIEN HANH TRICH XUAT CQT VA LUU FP16 VAO FOLDER OUTPUT
# ==========================================================
# Thiet lap folder xuat ban rieng biet de de dang quan ly va tai ve
EXPORT_DIR = WORKSPACE_DIR / 'export_qbh_cqt_cache_v6_1'
OUTPUT_CACHE_DIR = EXPORT_DIR / 'qbh_cqt_cache_v6_1'
FEATURE_DIR = OUTPUT_CACHE_DIR / 'features'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

index = []
feature_count = 0
start_time = time.perf_counter()

print(f'[START] Bat dau trich xuat CQT cho {len(inventory)} files tren GPU T4...')
for path in tqdm(inventory, desc='Extracting CQT FP16'):
    rel = path.relative_to(DATASET_ROOT).as_posix()
    source_hash = digest_file(path)
    audio = prep.load_audio(str(path))
    variants = [('base', audio)]

    is_query = '/1/' in ('/' + rel) and not path.name.lower().startswith('reference')
    if is_query and source_by_path.get(rel) == 'original':
        seed = int(source_hash[:16], 16)
        variants.append(('degraded_8khz_8bit', low_band_8bit(audio, seed=seed)))

    for variant, waveform in variants:
        chunks = []
        for start in chunk_starts(len(waveform)):
            chunk = waveform[start:start + CHUNK_SAMPLES]
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))
            feature = extractor.extract(chunk)
            if feature.shape != (1, 96, 83) or not torch.isfinite(feature).all():
                raise ValueError(f'Invalid CQT shape/value for {rel} ({variant}): {tuple(feature.shape)}')
            chunks.append(feature.detach().cpu().to(torch.float16).contiguous())

        identity = dict(path=rel, source_sha256=source_hash, variant=variant, preprocessing=PREPROCESS_CONFIG)
        key = hashlib.sha256(json.dumps(identity, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        feature_rel = f'features/{key}.pt'
        feature_path = OUTPUT_CACHE_DIR / feature_rel
        torch.save(chunks, feature_path)

        index.append(dict(relative_path=rel, source_sha256=source_hash, variant=variant,
                          feature_file=feature_rel, feature_sha256=digest_file(feature_path), chunks=len(chunks),
                          tensor_shape=[1, 96, 83], dtype='float16', cache_key=key))
        feature_count += len(chunks)

elapsed = time.perf_counter() - start_time
print(f'\n[OK] Hoan thanh trich xuat trong {elapsed:.1f} giay ({round(elapsed/60, 2)} phut)!')
print(f'[INFO] Tong so audio files      : {len(inventory)}')
print(f'[INFO] Tong so feature variants : {len(index)}')
print(f'[INFO] Tong so chunk CQT (FP16) : {feature_count}')

In [ ]:
# ==========================================================
# 7. TAO MANIFEST VA DONG GOI ZIP VAO FOLDER OUTPUT
# ==========================================================
manifest = dict(format_version=1, pipeline='qbh_final_v6_1', device=str(device),
                cqt_backend_sha256=EXPECTED_BACKEND_SHA,
                preprocessing=PREPROCESS_CONFIG, includes_test_or_holdout=False,
                audio_files=len(inventory), feature_variants=len(index), feature_chunks=feature_count,
                features=index)

manifest_path = OUTPUT_CACHE_DIR / 'qbh_cqt_manifest_v6_1.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'[OK] Da luu manifest tai: {manifest_path}')

# Nen folder qbh_cqt_cache_v6_1 thanh file ZIP dat truc tiep trong EXPORT_DIR va WORKSPACE_DIR
zip_output_path = EXPORT_DIR / 'qbh_cqt_cache_v6_1.zip'
print(f'[INFO] Dang dong goi file ZIP: {zip_output_path}...')
with zipfile.ZipFile(zip_output_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for file_path in OUTPUT_CACHE_DIR.rglob('*'):
        if file_path.is_file():
            z.write(file_path, file_path.relative_to(OUTPUT_CACHE_DIR).as_posix())

# Sao chep them mot ban tai /content de tien thao tac neu can
shutil.copyfile(zip_output_path, WORKSPACE_DIR / 'qbh_cqt_cache_v6_1.zip')

zip_size_mb = round(zip_output_path.stat().st_size / (1024**2), 2)
print(f'[OK] Dong goi thanh cong! Dung luong ZIP: {zip_size_mb} MB ({round(zip_size_mb/1024, 2)} GB)')
print(f'[INFO] Folder xuat ban tai: {EXPORT_DIR}')
print(f'       - Thu muc uncompressed: {OUTPUT_CACHE_DIR}')
print(f'       - File nen ZIP        : {zip_output_path}')

In [ ]:
# ==========================================================
# 8. SAO LUU VAO GOOGLE DRIVE VA TAI VE MAY TINH
# ==========================================================
from google.colab import files

# 1. Sao chep toan bo folder xuat ban va file ZIP vao Google Drive neu da ket noi
if DRIVE_MOUNTED:
    drive_export_folder = Path('/content/drive/MyDrive/export_qbh_cqt_cache_v6_1')
    drive_export_folder.mkdir(parents=True, exist_ok=True)
    drive_zip_path = drive_export_folder / 'qbh_cqt_cache_v6_1.zip'
    print(f'[INFO] Dang sao chep file ZIP vao Google Drive: {drive_zip_path}...')
    shutil.copyfile(zip_output_path, drive_zip_path)
    print(f'[OK] Da luu thanh cong vao Google Drive tai: {drive_export_folder}')

# 2. Kich hoat trinh duyet tai truc tiep file ZIP ve may tinh ca nhan
print('[INFO] Bat dau kich hoat tai file qbh_cqt_cache_v6_1.zip ve may tinh...')
try:
    files.download(str(zip_output_path))
    print('[OK] Da gui lenh tai ve qua trinh duyet thanh cong!')
except Exception as e:
    print(f'[INFO] Khong kich hoat tai tu dong duoc: {e}')
    print('[HUONG DAN] Ban co the mo tab Files ben trai Colab, vao thu muc export_qbh_cqt_cache_v6_1, click chuot phai vao qbh_cqt_cache_v6_1.zip va chon Download.')

### Huong dan su dung file Cache tren Kaggle:

1. Tai file `qbh_cqt_cache_v6_1.zip` (tu folder `export_qbh_cqt_cache_v6_1`) len **Kaggle** -> Tao **New Dataset** dat ten la: `qbh-cqt-cache-v6-1`.
2. Mo notebook huan luyen `conformer_bytehum_qbh_final_v6_1_kaggle.ipynb` tren Kaggle.
3. Bam **+ Add Input** -> Chon dataset `qbh-cqt-cache-v6-1` vua tao.
4. Bam **Run All**:
   - Notebook se tu dong phat hien `qbh_cqt_manifest_v6_1.json`, kiem tra ma bam SHA-256 va nap cache ngay lap tuc.
   - Khong con phai cho CPU Kaggle tinh CQT o moi epoch.